# 6. Production over Time

In [2]:
import pulp as pl

def solve(model):
    model.solve(pl.PULP_CBC_CMD(msg=False))
    if pl.LpStatus[model.status] != "Optimal":
        raise RuntimeError(pl.LpStatus[model.status])

demand = [7000, 15000, 10000, 8000]
model = pl.LpProblem("Production_over_Time", pl.LpMinimize)
R = [pl.LpVariable(f"regular_{t+1}", 0, 10000) for t in range(4)]
O = [pl.LpVariable(f"overtime_{t+1}", 0, 2500) for t in range(4)]
I = [pl.LpVariable(f"inventory_{t+1}", lowBound=0) for t in range(4)]
model += pl.lpSum(2000*R[t] + 2200*O[t] + 100*I[t] for t in range(4))
for t in range(4):
    model += (5000 if t == 0 else I[t-1]) + R[t] + O[t] - demand[t] == I[t]
model += I[3] == 0
solve(model)
print("Quarter   Regular   Overtime   Ending inventory")
for t in range(4):
    print(f"{t+1:7} {R[t].value():9.0f} {O[t].value():10.0f} {I[t].value():18.0f}")
print(f"Minimum total cost: USD {pl.value(model.objective):,.2f}")

Quarter   Regular   Overtime   Ending inventory
      1      7000          0               5000
      2     10000          0                  0
      3     10000          0                  0
      4      8000          0                  0
Minimum total cost: USD 70,500,000.00


Regular production by quarter: **7,000; 10,000; 10,000; 8,000**. No overtime is needed. Ending inventories: **5,000; 0; 0; 0**.

Minimum total cost: **$70,500,000**.